# K_01c – Animierte Energiefluss-Karten

**Grid-Arbitrage** · Animierte Visualisierung: Erzeugung, Verbrauch, Zonenfluss, Cross-Border

**Gruppe:** SC26_Gruppe_2 | **Status:** Kür – explorativ | **Abhängig von:** K_01 (Cache), K_01b (Zonen)

---

> 🎬 **5 GIF-Typen × 2 Zeitskalen = 10 Animationen**
>
> | GIF | Inhalt | Tagesverlauf | Jahresverlauf |
> |-----|--------|:---:|:---:|
> | A | Erzeugungsschwankungen (Dots nach ET, Grösse = MW) | 24h × 4 Jahreszeiten | 52 Wochen |
> | B | Verbrauchsschwankungen (Dots nach Zone, Grösse = Last) | 24h × 4 Jahreszeiten | 52 Wochen |
> | C | Zonenfluss (Moving Dots auf Pfaden, Dichte = MW) | 24h | 52 Wochen |
> | D | Cross-Border Import/Export (Dots an Grenzen) | 24h | 52 Wochen |
> | E | Kombiniert C + D | 24h | 52 Wochen |
>
> ⚠️ **Alle Flussdaten sind modelliert (synthetisch)** — keine Swissgrid-Messdaten.
> Vollständige Warnung in Sektion 2.

*Voraussetzung: K_01 ausgeführt (BFE/BFS Cache). K_01b für Zonenfarben (optional — Fallback vorhanden).*


| [← K_01b Zonenmodell](K_01b_Zonenmodell_Erweitert.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_01c'></a>

1. [Initialisierung](#init_K_01c)
2. [Warnung: Synthetische Flussdaten](#warnung_K_01c)
3. [Geodaten & Zonengeometrie](#geodaten_K_01c)
4. [Datenprofil-Modell](#profile_K_01c)
5. [Helper-Funktionen](#helper_K_01c)
6. [GIF A — Erzeugungsschwankungen](#gif_a_K_01c)
7. [GIF B — Verbrauchsschwankungen](#gif_b_K_01c)
8. [GIF C — Zonenfluss](#gif_c_K_01c)
9. [GIF D — Cross-Border](#gif_d_K_01c)
10. [GIF E — Kombiniert (C + D)](#gif_e_K_01c)
11. [Abschluss](#abschluss_K_01c)


---
## 1. Initialisierung<a id='init_K_01c'></a>

[↑ TOC](#toc_K_01c)

In [ ]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import os, warnings, json
import subprocess, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from matplotlib import cm
from matplotlib.animation import FuncAnimation, PillowWriter
import requests
warnings.filterwarnings('ignore')

for pkg in [('geopandas','geopandas'),('scipy','scipy'),('pillow','PIL')]:
    try: __import__(pkg[1])
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg[0],'-q'])

import geopandas as gpd

print(f"Pandas     : {pd.__version__}")
print(f"GeoPandas  : {gpd.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Matplotlib : {plt.matplotlib.__version__}")
print(f"📅 Ausgeführt: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


In [ ]:
# ── Config + Farben (SSOT — nur lesend) ─────────────────────────────────────
with open('../sync/config.json') as _f: CFG = json.load(_f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
CHARTS_DIR = os.path.join('../output', 'charts', SZ_AKTIV)
DATA_DIR   = os.path.join('../data', 'raw')
INTER_DIR  = os.path.join('../data', 'intermediate')
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI_GIF    = 130   # ⚙ Animationsauflösung (höher = grössere Dateien)
FPS_TAG    = 6     # ⚙ Frames/Sekunde Tagesverlauf (24 Frames → ~4s Loop)
FPS_JAHR   = 8     # ⚙ Frames/Sekunde Jahresverlauf (52 Frames → ~6.5s Loop)

_viz         = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK      = _viz.get('bg_dark',    '#0d1117')
BG_PANEL     = _viz.get('bg_panel',   '#141414')
C_LOAD       = _viz.get('c_load',     '#66BB6A')
C_CHARGE     = _viz.get('c_charge',   '#1565C0')
C_FEED       = _viz.get('c_feed',     '#B71C1C')
C_PRICE      = _viz.get('c_price',    '#FFA726')
C_SPINE      = _viz.get('c_spine',    '#333333')
C_ACHSE      = _viz.get('c_achse',    '#aaaaaa')
C_TICK       = _viz.get('c_tick',     '#bbbbbb')
C_LEGENDE_BG = _viz.get('c_legende_bg','#111111')
C_SOLAR      = _viz.get('c_solar',    '#FDD835')
C_GRENZWERT  = _viz.get('c_amber_dark','#FF6F00')
C_GRUEN_DARK = _viz.get('c_gruen_dark','#388E3C')
C_CYAN       = _viz.get('c_cyan',     '#26C6DA')
_stil        = CFG.get('visualisierung',{}).get('stil',{})
FS_TITEL     = _stil.get('schriftgroesse_titel', 13)
FS_KLEIN     = _stil.get('schriftgroesse_klein',  7)

# K_01b Zonenfarben (Fallback wenn K_01b nicht ausgeführt)
ZONE_COLORS_B = {
    'Nordost'        : '#1565C0',
    'Ostschweiz'     : '#00838F',
    'Mitte-Erzeugung': '#7B1FA2',
    'Mitte-Transit'  : '#388E3C',
    'West'           : '#FF6F00',
    'Süd'            : '#B71C1C',
}
ET_COLORS = {
    'Solar':      C_SOLAR,      'Wasserkraft': C_CHARGE,
    'Kernkraft':  '#7B1FA2',    'Wind':        C_CYAN,
    'Biomasse':   C_GRUEN_DARK, 'Andere':      '#9E9E9E',
}
print(f"Config geladen | Szenario: {SZ_AKTIV} | Charts: {CHARTS_DIR}")


---
## 2. Warnung: Synthetische Flussdaten<a id='warnung_K_01c'></a>

[↑ TOC](#toc_K_01c)

---

> ## ⚠️ ALLE ENERGIEFLÜSSE IN DIESEM NOTEBOOK SIND MODELLIERT — KEINE MESSDATEN
>
> ### Was ist synthetisch:
> - **Zonenflüsse** (GIF C/E): Aus Imbalance-Differenzen abgeleitet, kein echter Lastfluss-Solver
> - **Stündliche Profile**: Gauss-/Sinus-Kurven nach Energieträgertypik — keine ENTSO-E Stundendaten
> - **Jahresverlauf**: Saisonale Kapazitätsfaktoren (BFE 2023) × synthetisches Wochenprofil
> - **Cross-Border** (GIF D/E): Netto-Imbalance der CH-Zonen als Proxy für Grenzflüsse
>
> ### Was real ist:
> - **Installierte Kapazitäten** pro Zone aus BFE GeoPackage (322k Anlagen)
> - **Zonenstruktur** aus K_01b (6-Zonen-Modell)
> - **Kapazitätsfaktoren** (BFE Elektrizitätsstatistik 2023)
> - **Geographische Positionen** (Zone-Zentroide aus BFS Kantonsdaten)
>
> ### Zweck dieser Animationen:
> Strukturelle Muster sichtbar machen (Duck Curve, Winter-Peak, Süd→Nord-Fluss),
> **nicht** operative Lastfluss-Simulation. Für Investitionsentscheide:
> Swissgrid SCADA-Daten oder professionellen Netzplaner hinzuziehen.

---


---
## 3. Geodaten & Zonengeometrie<a id='geodaten_K_01c'></a>

[↑ TOC](#toc_K_01c)

In [ ]:
# ── Kantonsgrenzen laden (K_01-Cache: kantone.gpkg) ──────────────────────────
KANT_CANDIDATES = [
    os.path.join(DATA_DIR, 'kantone.gpkg'),              # K_01 Standard-Output
    os.path.join(DATA_DIR, 'swissboundaries3d.gpkg'),
    os.path.join(DATA_DIR, 'swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.gpkg'),
    os.path.join('..', 'data', 'raw', 'kantone.gpkg'),
]
KANT_NUM_TO_ABK = {
    1:'ZH',2:'BE',3:'LU',4:'UR',5:'SZ',6:'OW',7:'NW',8:'GL',
    9:'ZG',10:'FR',11:'SO',12:'BS',13:'BL',14:'SH',15:'AR',16:'AI',
    17:'SG',18:'GR',19:'AG',20:'TG',21:'TI',22:'VD',23:'VS',24:'NE',25:'GE',26:'JU',
}

gdf_kant = None
for path in KANT_CANDIDATES:
    if os.path.exists(path) and os.path.getsize(path) > 50_000:
        try:
            layers = gpd.list_layers(path)
            lname = next((l for l in layers['name'] if 'kanton' in l.lower()),
                         layers['name'].iloc[0])
            gdf_raw = gpd.read_file(path, layer=lname)
            if gdf_raw.crs and gdf_raw.crs.to_epsg() != 4326:
                gdf_raw = gdf_raw.to_crs(epsg=4326)
            # Kürzel aus Spalte ermitteln: icc (lowercase), KANTONSNUM, name, etc.
            kab_col = None
            for col in gdf_raw.columns:
                sample = str(gdf_raw[col].dropna().iloc[0]) if len(gdf_raw)>0 else ''
                if col.lower() in ['icc','kab','abbreviation','abb']:
                    kab_col = col; break
                if col.lower() == 'kantonsnum':
                    kab_col = col; break
            if kab_col is None:
                for col in gdf_raw.columns:
                    vals = gdf_raw[col].dropna().astype(str).str.strip()
                    if vals.str.len().between(2,3).mean() > 0.8:
                        kab_col = col; break
            if kab_col:
                s = gdf_raw[kab_col].astype(str).str.strip()
                # Numerische Kantone (KANTONSNUM 1-26)
                if s.str.isnumeric().all():
                    gdf_raw['KAB'] = s.astype(int).map(KANT_NUM_TO_ABK)
                else:
                    # icc → uppercase (z.B. 'zh' → 'ZH')
                    gdf_raw['KAB'] = s.str.upper().str[:2]
                gdf_kant = gdf_raw[gdf_raw['KAB'].notna()].copy()
                print(f"Kantone geladen: {len(gdf_kant)} | Datei: {os.path.basename(path)} | Spalte: {kab_col}")
                break
        except Exception as e:
            print(f"  {os.path.basename(path)}: {e}")

if gdf_kant is None:
    print("⚠️  Kantonsgrenzen nicht gefunden — Animationen laufen mit CH-Umriss-Fallback.")

# Fallback: vereinfachter CH-Umriss als Polygon (WGS84, handdigitalisiert)
CH_OUTLINE = [
    (5.96,47.81),(6.44,47.87),(7.19,47.97),(7.61,47.59),(8.23,47.69),
    (8.62,47.76),(9.00,47.69),(9.53,47.52),(10.49,47.40),(10.38,47.01),
    (9.97,46.65),(9.56,46.49),(9.08,46.12),(8.86,46.07),(8.51,46.00),
    (8.08,46.11),(7.58,46.02),(7.06,45.92),(6.74,46.10),(6.34,46.44),
    (5.97,46.80),(5.96,47.17),(5.96,47.81)
]

print(f"Fallback CH-Umriss: {len(CH_OUTLINE)} Punkte")


In [ ]:
# ── Zonen-Zentroide & Geometrien ─────────────────────────────────────────────
# Handkalibrierte Zentroide der 6 K_01b-Zonen (WGS84)
# ⚙ Könnten aus gdf_kant automatisch berechnet werden (nach Config-Integration)
ZONE_CENTERS = {
    'Nordost'        : (8.68, 47.38),   # Zürich
    'Ostschweiz'     : (9.40, 46.78),   # Chur
    'Mitte-Erzeugung': (8.08, 47.40),   # Aarau
    'Mitte-Transit'  : (7.52, 46.95),   # Bern
    'West'           : (6.65, 46.55),   # Lausanne
    'Süd'            : (7.95, 46.25),   # Visp/Gotthard
}

# Cross-Border Grenzpunkte
BORDER_POINTS = {
    'DE': (7.60, 47.72),   # Basel
    'AT': (9.55, 47.48),   # Rheintal
    'IT': (8.95, 45.92),   # Chiasso / Domodossola
    'FR': (6.10, 46.22),   # Genf / Jura
}
BORDER_COLORS = {'DE':'#42A5F5','AT':'#EF5350','IT':'#66BB6A','FR':'#FFA726'}

# Interne Zonenfluss-Pfade (von_Zone, nach_Zone, Beschriftung)
# Richtung = normal; negatives MW = umgekehrt
ZONE_PAIRS = [
    ('Süd',             'Nordost',         'Göschenen-Airolo'),
    ('Süd',             'Mitte-Transit',   'Gotthard West'),
    ('Mitte-Erzeugung', 'Nordost',         'AKW-Export'),
    ('Mitte-Erzeugung', 'Mitte-Transit',   'Interne Verteilung'),
    ('Ostschweiz',      'Nordost',         'Ostschweiz→ZH'),
    ('West',            'Mitte-Transit',   'VD↔BE'),
]

# Zone-Polygon-Union (für farbige Hintergrundfelder)
gdf_zones_geo = None
KANTON_TO_ZONE_B = {
    'ZH':'Nordost','TG':'Nordost','SH':'Nordost','SG':'Nordost',
    'AR':'Ostschweiz','AI':'Ostschweiz','GR':'Ostschweiz','GL':'Ostschweiz','SZ':'Ostschweiz',
    'AG':'Mitte-Erzeugung','SO':'Mitte-Erzeugung','BL':'Mitte-Erzeugung',
    'BE':'Mitte-Transit','LU':'Mitte-Transit','BS':'Mitte-Transit',
    'ZG':'Mitte-Transit','NW':'Mitte-Transit','OW':'Mitte-Transit',
    'VD':'West','GE':'West','NE':'West','JU':'West','FR':'West',
    'VS':'Süd','TI':'Süd','UR':'Süd',
}

if gdf_kant is not None:
    try:
        # KAB = 2-letter canton code; try multiple column names
        kab_col = 'KAB' if 'KAB' in gdf_kant.columns else next((c for c in gdf_kant.columns if c.upper() in ['KANTONSNUM','ABBREVIATION','ABB']), None)
        gdf_kant['Zone_B'] = gdf_kant[kab_col].map(KANTON_TO_ZONE_B)
        gdf_zones_geo = (gdf_kant.dropna(subset=['Zone_B'])
                         .dissolve(by='Zone_B', as_index=False)
                         [['Zone_B','geometry']])
        print(f"Zonen-Polygone: {len(gdf_zones_geo)} Zonen aufgebaut")
    except Exception as e:
        print(f"Zonen-Union fehlgeschlagen: {e} — nur Zentroide werden genutzt")

print("Zentroide:")
for z, pos in ZONE_CENTERS.items():
    print(f"  {z:<16}: ({pos[0]:.2f}°E, {pos[1]:.2f}°N)")


---
## 4. Datenprofil-Modell<a id='profile_K_01c'></a>

[↑ TOC](#toc_K_01c)

Synthetische stündliche und wöchentliche Profile. Alle Ausgaben in MW.
⚙ = Parameter die nach Config-Integration aus `config.json` kämen.

In [ ]:
# ── Zonenbasisdaten (aus K_01b-Logik) ────────────────────────────────────────
KANTON_POP = {
    'ZH':1593583,'BE':1065607,'LU':433654,'UR':37208,'SZ':165539,'OW':39028,
    'NW':43921,'GL':40590,'ZG':130426,'FR':340765,'SO':279375,'BS':183709,
    'BL':297025,'SH':86928,'AR':58050,'AI':16293,'SG':530468,'GR':202461,
    'AG':718084,'TG':295373,'TI':358353,'VD':826400,'VS':357043,'NE':177589,
    'GE':511655,'JU':73584
}
SPEZ_KW = 0.76  # ⚙ kW Mittellast/Person

# Mittlere Verbrauch + geschätzte Produktion pro Zone [MW]
ZONE_BASE = {}
for z in ZONE_COLORS_B:
    pop = sum(KANTON_POP.get(k,0) for k,v in KANTON_TO_ZONE_B.items() if v==z)
    ZONE_BASE[z] = {'pop': pop, 'verbrauch_mw': pop * SPEZ_KW / 1000}

# Geschätzte installierte Kapazitäten [MW installiert] — aus BFE-Kenntnis
# ⚙ Würde nach Config-Integration aus gdf_plants berechnet
ZONE_PROD_INSTALLED = {
    'Nordost':         {'Solar':1800,'Wasserkraft': 500,'Kernkraft':   0,'Andere': 200},
    'Ostschweiz':      {'Solar': 600,'Wasserkraft':2800,'Kernkraft':   0,'Andere': 100},
    'Mitte-Erzeugung': {'Solar':1200,'Wasserkraft': 800,'Kernkraft':5000,'Andere': 300},
    'Mitte-Transit':   {'Solar':1400,'Wasserkraft':1200,'Kernkraft':   0,'Andere': 400},
    'West':            {'Solar':1000,'Wasserkraft': 900,'Kernkraft':   0,'Andere': 150},
    'Süd':             {'Solar': 800,'Wasserkraft':8500,'Kernkraft':   0,'Andere':  50},
}
CF = {'Solar':0.12,'Wasserkraft':0.38,'Kernkraft':0.80,'Andere':0.40}

SAISONS = ['Winter','Frühling','Sommer','Herbst']
# Saisonale Kapazitätsfaktoren-Skalierung je ET
CF_SEASONAL = {
    'Solar':      {'Winter':0.30,'Frühling':0.75,'Sommer':1.00,'Herbst':0.50},
    'Wasserkraft':{'Winter':0.65,'Frühling':0.95,'Sommer':0.90,'Herbst':0.72},
    'Kernkraft':  {'Winter':1.00,'Frühling':0.98,'Sommer':0.92,'Herbst':0.98},
    'Andere':     {'Winter':1.10,'Frühling':1.00,'Sommer':0.85,'Herbst':1.00},
}
VERBRAUCH_SAISONAL = {'Winter':1.15,'Frühling':1.00,'Sommer':0.87,'Herbst':1.02}

HOURS = np.arange(24)
WEEKS = np.arange(52)

def solar_h(h):
    '''Solar-Tagesprofil normiert [0-1]'''
    return np.maximum(0.0, np.sin(np.pi * (h - 6.0) / 13.0))

def hydro_h(h):
    '''Wasserkraft-Tagesprofil (Dispatch-Charakteristik) normiert [0.7-1.3]'''
    return (1.0 + 0.22*np.exp(-((h-10.5)**2)/14)
                + 0.18*np.exp(-((h-19.0)**2)/10)
                - 0.10*np.exp(-((h-4.0)**2)/4))

def nuclear_h(h):
    return np.ones_like(np.array(h, dtype=float))

def consumption_h(h, saison='Frühling'):
    s = VERBRAUCH_SAISONAL.get(saison, 1.0)
    return s * (1.0 + 0.38*np.exp(-((h-8.5)**2)/4.5)
                    + 0.44*np.exp(-((h-19.0)**2)/5.0)
                    - 0.28*np.exp(-((h-4.0)**2)/3.0))

def zone_prod_mw(zone, h, saison):
    '''MW Produktion einer Zone bei Stunde h, Jahreszeit saison'''
    total = 0.0
    for et, inst_mw in ZONE_PROD_INSTALLED.get(zone, {}).items():
        cf_base = CF.get(et, 0.4)
        cf_s    = CF_SEASONAL.get(et, {}).get(saison, 1.0)
        if et == 'Solar':
            profile = solar_h(h)
        elif et == 'Wasserkraft':
            profile = hydro_h(h)
        elif et == 'Kernkraft':
            profile = nuclear_h(h)
        else:
            profile = 0.85  # Andere: flat
        total += inst_mw * cf_base * cf_s * profile
    return total

def zone_cons_mw(zone, h, saison):
    return ZONE_BASE[zone]['verbrauch_mw'] * consumption_h(h, saison)

def zone_imbalance_mw(zone, h, saison):
    return zone_prod_mw(zone, h, saison) - zone_cons_mw(zone, h, saison)

# Jahresprofil (52 Wochen) — saisonaler Verlauf
def solar_week(w):    return 0.12 + 0.10*np.sin(2*np.pi*(w-12)/52)
def hydro_week(w):    return 0.38 + 0.14*np.sin(2*np.pi*(w-15)/52)
def nuclear_week(w):  return 0.79 - 0.05*np.sin(2*np.pi*(w-26)/52)  # Sommerrevision
def cons_week(w):     return 1.00 - 0.14*np.sin(2*np.pi*(w-12)/52)  # Sommer-Tief

def zone_prod_mw_week(zone, w):
    total = 0.0
    for et, inst_mw in ZONE_PROD_INSTALLED.get(zone, {}).items():
        if et == 'Solar':        cf = solar_week(w)
        elif et == 'Wasserkraft': cf = hydro_week(w)
        elif et == 'Kernkraft':  cf = nuclear_week(w)
        else:                    cf = 0.40
        total += inst_mw * cf
    return total

def zone_cons_mw_week(zone, w):
    return ZONE_BASE[zone]['verbrauch_mw'] * cons_week(w)

def zone_imbalance_mw_week(zone, w):
    return zone_prod_mw_week(zone, w) - zone_cons_mw_week(zone, w)

# Precompute für Performance
print("Precomputing Stundendaten...")
HOURLY = {}
for saison in SAISONS:
    HOURLY[saison] = {}
    for zone in ZONE_COLORS_B:
        HOURLY[saison][zone] = {
            'prod': [zone_prod_mw(zone, h, saison) for h in HOURS],
            'cons': [zone_cons_mw(zone, h, saison) for h in HOURS],
            'imb':  [zone_imbalance_mw(zone, h, saison) for h in HOURS],
        }

print("Precomputing Jahresverlauf...")
WEEKLY = {}
for zone in ZONE_COLORS_B:
    WEEKLY[zone] = {
        'prod': [zone_prod_mw_week(zone, w) for w in WEEKS],
        'cons': [zone_cons_mw_week(zone, w) for w in WEEKS],
        'imb':  [zone_imbalance_mw_week(zone, w) for w in WEEKS],
    }

print("✅ Datenprofile bereit")
print(f"   Zonen: {list(ZONE_COLORS_B.keys())}")
print(f"   Jahreszeiten: {SAISONS}")
print(f"   Stunden: {len(HOURS)} | Wochen: {len(WEEKS)}")


---
## 5. Helper-Funktionen<a id='helper_K_01c'></a>

[↑ TOC](#toc_K_01c)

In [ ]:
# ── draw_base_map: Kantone + Zonen-Flächen ───────────────────────────────────
MAP_XLIM = (5.88, 10.60)
MAP_YLIM = (45.78, 47.92)

def draw_base_map(ax, zone_alpha=0.12):
    '''Zeichnet Karte: dunkler BG, Zonen-Fläche, Kantonsgrenzen.'''
    ax.set_xlim(*MAP_XLIM); ax.set_ylim(*MAP_YLIM)
    ax.set_facecolor('#090d14'); ax.set_axis_off()

    if gdf_zones_geo is not None:
        for _, row in gdf_zones_geo.iterrows():
            z = row['Zone_B']
            col = ZONE_COLORS_B.get(z, '#555')
            gpd.GeoDataFrame([row], geometry='geometry', crs='EPSG:4326').plot(
                ax=ax, color=col, alpha=zone_alpha, linewidth=0, zorder=1)

    if gdf_kant is not None:
        gdf_kant.boundary.plot(ax=ax, color='#2a3040', linewidth=0.6, zorder=2)
        try:
            ch_outer = gdf_kant.unary_union.boundary
            gpd.GeoDataFrame(geometry=[ch_outer], crs='EPSG:4326').plot(
                ax=ax, color='#445566', linewidth=1.8, zorder=3)
        except Exception:
            pass
    else:
        # Fallback: vereinfachter CH-Umriss
        from matplotlib.patches import Polygon as MplPolygon
        from matplotlib.collections import PatchCollection
        xs = [p[0] for p in CH_OUTLINE]
        ys = [p[1] for p in CH_OUTLINE]
        ax.plot(xs, ys, color='#445566', linewidth=1.5, zorder=3)
        ax.fill(xs, ys, color='#0d1520', alpha=0.5, zorder=1)


def add_zone_labels(ax, fontsize=7):
    '''Zone-Labels an Zentroiden.'''
    for z, (lon, lat) in ZONE_CENTERS.items():
        short = z.replace('Mitte-','M-').replace('Ostschweiz','Ostschw.')
        ax.text(lon, lat+0.08, short, ha='center', va='bottom',
                color=ZONE_COLORS_B[z], fontsize=fontsize, fontweight='bold',
                zorder=15, bbox=dict(boxstyle='round,pad=0.15', fc='#090d14', alpha=0.7, lw=0))


def draw_flow_dots(ax, zone_from, zone_to, mw_flow, frame, n_frames,
                   min_mw=80, base_dots=3, max_dots=10, base_size=35, max_size=150):
    '''Moving dots entlang Pfad von zone_from nach zone_to.
    Richtung: positive mw = from->to, negativ = to->from.
    Dichte & Grösse proportional zum MW-Betrag.
    '''
    if abs(mw_flow) < min_mw:
        return
    p1 = ZONE_CENTERS[zone_from]
    p2 = ZONE_CENTERS[zone_to]
    if mw_flow < 0:
        p1, p2 = p2, p1

    n_dots   = min(max_dots, max(base_dots, int(abs(mw_flow) / 400)))
    dot_size = min(max_size, max(base_size, abs(mw_flow) / 12))
    col      = ZONE_COLORS_B[zone_from] if mw_flow >= 0 else ZONE_COLORS_B[zone_to]
    speed    = 0.8 + abs(mw_flow) / 3000   # schneller bei mehr MW

    for i in range(n_dots):
        t = (frame * speed / n_frames + i / n_dots) % 1.0
        x = p1[0] + t * (p2[0] - p1[0])
        y = p1[1] + t * (p2[1] - p1[1])
        ax.scatter(x, y, s=dot_size, c=col, alpha=0.88, zorder=10,
                   linewidths=0, rasterized=True)

    # Pfad-Linie (dezent)
    ax.plot([ZONE_CENTERS[zone_from][0], ZONE_CENTERS[zone_to][0]],
            [ZONE_CENTERS[zone_from][1], ZONE_CENTERS[zone_to][1]],
            color=col, lw=0.5, alpha=0.20, zorder=5)


def draw_border_dots(ax, neighbor, mw_flow, frame, n_frames,
                     ch_center=(8.20, 46.95), min_mw=50):
    '''Moving dots zwischen CH-Zentrum und Grenzpunkt.
    Positiv = CH exportiert; negativ = CH importiert.
    '''
    if abs(mw_flow) < min_mw:
        return
    bp  = BORDER_POINTS[neighbor]
    col = BORDER_COLORS[neighbor]
    if mw_flow >= 0:
        p_from, p_to = ch_center, bp   # Export
    else:
        p_from, p_to = bp, ch_center   # Import

    n_dots   = min(8, max(2, int(abs(mw_flow) / 500)))
    dot_size = min(120, max(25, abs(mw_flow) / 15))

    for i in range(n_dots):
        t = (frame * 0.9 / n_frames + i / n_dots) % 1.0
        x = p_from[0] + t * (p_to[0] - p_from[0])
        y = p_from[1] + t * (p_to[1] - p_from[1])
        ax.scatter(x, y, s=dot_size, c=col, alpha=0.90, zorder=11,
                   linewidths=0, rasterized=True)

    ax.plot([ch_center[0], bp[0]], [ch_center[1], bp[1]],
            color=col, lw=0.5, alpha=0.18, zorder=4)


def add_timestamp_bar(fig, label_left, label_center, label_right, alpha_center=1.0):
    '''Info-Leiste oben im Bild.'''
    fig.text(0.02, 0.97, label_left,   ha='left',   va='top', color='#aaaaaa', fontsize=8,
             transform=fig.transFigure)
    fig.text(0.50, 0.97, label_center, ha='center', va='top', color='white',
             fontsize=9, fontweight='bold', transform=fig.transFigure, alpha=alpha_center)
    fig.text(0.98, 0.97, label_right,  ha='right',  va='top', color='#ff5252', fontsize=7,
             transform=fig.transFigure)


def make_gif(update_fn, n_frames, fps, path, figsize=(14,9)):
    '''Generischer GIF-Builder. update_fn(fig, ax, frame, n_frames) -> None.'''
    fig = plt.figure(figsize=figsize, facecolor=BG_DARK)
    ax  = fig.add_subplot(111)

    def _update(frame):
        ax.clear()
        for txt in fig.texts: txt.set_visible(False)
        fig.texts.clear()
        update_fn(fig, ax, frame, n_frames)
        return []

    anim = FuncAnimation(fig, _update, frames=n_frames,
                         interval=int(1000/fps), blit=False)
    writer = PillowWriter(fps=fps)
    anim.save(path, writer=writer, dpi=DPI_GIF)
    plt.close(fig)
    print(f"✅ {os.path.basename(path)} ({n_frames} Frames, {fps} fps, "
          f"{os.path.getsize(path)//1024} KB)")


print("✅ Helper-Funktionen geladen")

MONAT_KURZ = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']


---
## 6. GIF A — Erzeugungsschwankungen<a id='gif_a_K_01c'></a>

[↑ TOC](#toc_K_01c)

Dots pro Zone & Energieträger. Grösse = aktuelle Produktions-MW. Tagesverlauf: 4× (je Jahreszeit, 24 Frames). Jahresverlauf: 52 Frames.

In [ ]:
# ── GIF A: Erzeugung Tagesverlauf (4 Jahreszeiten) ───────────────────────────
import itertools

# ET-spezifische Dot-Offset damit Dots nicht überlagern
ET_OFFSETS = {
    'Solar':      ( 0.08,  0.06),
    'Wasserkraft':(-0.08, -0.04),
    'Kernkraft':  ( 0.00,  0.12),
    'Andere':     ( 0.06, -0.10),
}

def update_gif_a_tag(fig, ax, frame, n_frames, saison):
    h = frame  # frame = Stunde 0-23
    draw_base_map(ax, zone_alpha=0.08)
    add_zone_labels(ax, fontsize=6)

    for zone, (cx, cy) in ZONE_CENTERS.items():
        for et, inst_mw in ZONE_PROD_INSTALLED.get(zone, {}).items():
            if et not in ET_COLORS: continue
            cf_base = CF.get(et, 0.4)
            cf_s    = CF_SEASONAL.get(et, {}).get(saison, 1.0)
            if et == 'Solar':       profile = float(solar_h(h))
            elif et == 'Wasserkraft': profile = float(hydro_h(h))
            elif et == 'Kernkraft':  profile = 1.0
            else:                   profile = 0.85
            mw_now = inst_mw * cf_base * cf_s * profile
            if mw_now < 50: continue
            dot_size = min(500, max(20, mw_now / 6))
            ox, oy = ET_OFFSETS.get(et, (0, 0))
            ax.scatter(cx+ox, cy+oy, s=dot_size, c=ET_COLORS[et],
                       alpha=0.80, zorder=12, linewidths=0, rasterized=True)

    # Legende
    handles = [mpatches.Patch(color=ET_COLORS[et], label=et)
               for et in ['Solar','Wasserkraft','Kernkraft','Andere']]
    ax.legend(handles=handles, loc='lower left', fontsize=6,
              framealpha=0.5, facecolor='#111', labelcolor='white')

    add_timestamp_bar(fig, f"⚙ Modelliert | {saison}",
                      f"Erzeugung pro Zone & Energieträger — {h:02d}:00 Uhr",
                      "⚠️ Synthetische Daten")

for saison in SAISONS:
    fname = f"kuer_k01c_gif_a_tag_{saison.lower()}.gif"
    path  = os.path.join(CHARTS_DIR, fname)
    make_gif(lambda fig,ax,fr,nf,s=saison: update_gif_a_tag(fig,ax,fr,nf,s),
             n_frames=24, fps=FPS_TAG, path=path)


In [ ]:
# ── GIF A: Erzeugung Jahresverlauf (52 Wochen) ───────────────────────────────
def update_gif_a_jahr(fig, ax, frame, n_frames):
    w = frame  # frame = Woche 0-51
    monat = int(w / 52 * 12) + 1
    monat_name = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez'][monat-1]
    draw_base_map(ax, zone_alpha=0.08)
    add_zone_labels(ax, fontsize=6)

    for zone, (cx, cy) in ZONE_CENTERS.items():
        for et, inst_mw in ZONE_PROD_INSTALLED.get(zone, {}).items():
            if et not in ET_COLORS: continue
            if et == 'Solar':        cf = float(solar_week(w))
            elif et == 'Wasserkraft': cf = float(hydro_week(w))
            elif et == 'Kernkraft':  cf = float(nuclear_week(w))
            else:                   cf = 0.40
            mw_now = inst_mw * cf
            if mw_now < 50: continue
            dot_size = min(600, max(15, mw_now / 7))
            ox, oy = ET_OFFSETS.get(et, (0,0))
            ax.scatter(cx+ox, cy+oy, s=dot_size, c=ET_COLORS[et],
                       alpha=0.80, zorder=12, linewidths=0, rasterized=True)

    handles = [mpatches.Patch(color=ET_COLORS[et], label=et)
               for et in ['Solar','Wasserkraft','Kernkraft','Andere']]
    ax.legend(handles=handles, loc='lower left', fontsize=6,
              framealpha=0.5, facecolor='#111', labelcolor='white')
    add_timestamp_bar(fig, "⚙ Modelliert | Ø Tageswert",
                      f"Erzeugung — Woche {w+1:02d} / 52 ({monat_name})",
                      "⚠️ Synthetische Daten")

path = os.path.join(CHARTS_DIR, 'kuer_k01c_gif_a_jahr.gif')
make_gif(update_gif_a_jahr, n_frames=52, fps=FPS_JAHR, path=path)


---
## 7. GIF B — Verbrauchsschwankungen<a id='gif_b_K_01c'></a>

[↑ TOC](#toc_K_01c)

Farbige Dots (Zonenfarbe) an Zonenzentroiden. Grösse = aktuelle Last-MW.

In [ ]:
# ── GIF B: Verbrauch Tagesverlauf ────────────────────────────────────────────
def update_gif_b_tag(fig, ax, frame, n_frames, saison):
    h = frame
    draw_base_map(ax, zone_alpha=0.06)

    for zone, (cx, cy) in ZONE_CENTERS.items():
        mw_cons = float(zone_cons_mw(zone, h, saison))
        mw_prod = float(zone_prod_mw(zone, h, saison))
        imb     = mw_prod - mw_cons
        dot_size = min(700, max(40, mw_cons / 4))
        col = ZONE_COLORS_B[zone]
        ax.scatter(cx, cy, s=dot_size, c=col, alpha=0.70, zorder=10,
                   linewidths=0, rasterized=True)
        # Imbalance-Ring: grün=Überschuss, rot=Defizit
        ring_col = '#66BB6A' if imb > 0 else '#EF5350'
        ring_size = min(800, max(80, abs(imb)/3))
        ax.scatter(cx, cy, s=ring_size, facecolors='none',
                   edgecolors=ring_col, linewidths=2.5, alpha=0.65, zorder=11)
        # MW-Label
        ax.text(cx, cy-0.14, f'{mw_cons:.0f} MW', ha='center', va='top',
                color='white', fontsize=5.5, zorder=13,
                bbox=dict(boxstyle='round,pad=0.1', fc='#111', alpha=0.6, lw=0))

    add_zone_labels(ax, fontsize=6)
    handles = [
        mpatches.Patch(color='#66BB6A', label='Überschuss'),
        mpatches.Patch(color='#EF5350', label='Defizit'),
    ]
    ax.legend(handles=handles, title='Imbalance-Ring', title_fontsize=6,
              loc='lower left', fontsize=6, framealpha=0.5,
              facecolor='#111', labelcolor='white')
    add_timestamp_bar(fig, f"⚙ Modelliert | {saison}",
                      f"Verbrauch je Zone (Dot) + Imbalance (Ring) — {h:02d}:00 Uhr",
                      "⚠️ Synthetische Daten")

for saison in SAISONS:
    fname = f"kuer_k01c_gif_b_tag_{saison.lower()}.gif"
    path  = os.path.join(CHARTS_DIR, fname)
    make_gif(lambda fig,ax,fr,nf,s=saison: update_gif_b_tag(fig,ax,fr,nf,s),
             n_frames=24, fps=FPS_TAG, path=path)


In [ ]:
# ── GIF B: Verbrauch Jahresverlauf ───────────────────────────────────────────
def update_gif_b_jahr(fig, ax, frame, n_frames):
    w = frame
    monat_name = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez'][int(w/52*12)]
    draw_base_map(ax, zone_alpha=0.06)

    for zone, (cx, cy) in ZONE_CENTERS.items():
        mw_cons = float(zone_cons_mw_week(zone, w))
        mw_prod = float(zone_prod_mw_week(zone, w))
        imb     = mw_prod - mw_cons
        dot_size = min(800, max(50, mw_cons / 4))
        ax.scatter(cx, cy, s=dot_size, c=ZONE_COLORS_B[zone], alpha=0.70, zorder=10,
                   linewidths=0, rasterized=True)
        ring_col  = '#66BB6A' if imb > 0 else '#EF5350'
        ring_size = min(900, max(80, abs(imb)/3))
        ax.scatter(cx, cy, s=ring_size, facecolors='none',
                   edgecolors=ring_col, linewidths=2.5, alpha=0.65, zorder=11)
        ax.text(cx, cy-0.14, f'{mw_cons:.0f} MW', ha='center', va='top',
                color='white', fontsize=5.5, zorder=13,
                bbox=dict(boxstyle='round,pad=0.1', fc='#111', alpha=0.6, lw=0))

    add_zone_labels(ax, fontsize=6)
    add_timestamp_bar(fig, "⚙ Modelliert | Ø Tageswert",
                      f"Verbrauch je Zone — Woche {w+1:02d}/52 ({monat_name})",
                      "⚠️ Synthetische Daten")

path = os.path.join(CHARTS_DIR, 'kuer_k01c_gif_b_jahr.gif')
make_gif(update_gif_b_jahr, n_frames=52, fps=FPS_JAHR, path=path)


---
## 8. GIF C — Zonenfluss<a id='gif_c_K_01c'></a>

[↑ TOC](#toc_K_01c)

Moving Dots auf Pfaden zwischen Zonen. Richtung = Flussrichtung. Dot-Dichte & Grösse = MW-Intensität.

In [ ]:
# ── GIF C: Zonenfluss Tagesverlauf ───────────────────────────────────────────
def update_gif_c(fig, ax, frame, n_frames, get_imb_fn, label):
    '''Zonenfluss-Animation. get_imb_fn(zone) -> imbalance MW dieser Stunde/Woche.'''
    draw_base_map(ax, zone_alpha=0.10)

    # Pfade zwischen Zonen
    for zone_from, zone_to, path_label in ZONE_PAIRS:
        imb_from = get_imb_fn(zone_from)
        imb_to   = get_imb_fn(zone_to)
        # Fluss = Überschuss der Exportzone in Richtung Defizit-Zone
        if imb_from > 0 and imb_to < 0:
            mw_flow = min(abs(imb_from), abs(imb_to))
            draw_flow_dots(ax, zone_from, zone_to, mw_flow, frame, n_frames)
        elif imb_to > 0 and imb_from < 0:
            mw_flow = min(abs(imb_to), abs(imb_from))
            draw_flow_dots(ax, zone_to, zone_from, mw_flow, frame, n_frames)
        else:
            # Beide positiv: grösserer Überschuss exportiert
            net = imb_from - imb_to
            if abs(net) > 100:
                draw_flow_dots(ax, zone_from, zone_to, net, frame, n_frames)

    # Statische Zone-Dots (Imbalance)
    for zone, (cx, cy) in ZONE_CENTERS.items():
        imb = get_imb_fn(zone)
        col = '#66BB6A' if imb > 0 else '#EF5350'
        ax.scatter(cx, cy, s=max(60, min(400, abs(imb)/4)), c=col, alpha=0.60,
                   zorder=8, linewidths=0)
        ax.text(cx, cy, f'{imb:+.0f}', ha='center', va='center',
                color='white', fontsize=5, fontweight='bold', zorder=14)

    add_zone_labels(ax, fontsize=6)
    add_timestamp_bar(fig, "⚙ Modelliert — kein Lastfluss-Solver",
                      label, "⚠️ Synthetische Daten")

# Tagesverlauf: 4 Jahreszeiten
for saison in SAISONS:
    def make_fn(s):
        def fn(fig, ax, frame, n_frames):
            get_imb = lambda z: float(HOURLY[s][z]['imb'][frame])
            update_gif_c(fig, ax, frame, n_frames, get_imb,
                         f"Zonenfluss — {frame:02d}:00 Uhr ({s})")
        return fn
    path = os.path.join(CHARTS_DIR, f"kuer_k01c_gif_c_tag_{saison.lower()}.gif")
    make_gif(make_fn(saison), n_frames=24, fps=FPS_TAG, path=path)


In [ ]:
# ── GIF C: Zonenfluss Jahresverlauf ──────────────────────────────────────────
def update_gif_c_jahr(fig, ax, frame, n_frames):
    w = frame
    monat = MONAT_KURZ[int(w/52*12)]
    get_imb = lambda z: float(WEEKLY[z]['imb'][w])
    update_gif_c(fig, ax, frame, n_frames, get_imb,
                 f"Zonenfluss — Woche {w+1:02d}/52 ({monat})")

path = os.path.join(CHARTS_DIR, 'kuer_k01c_gif_c_jahr.gif')
make_gif(update_gif_c_jahr, n_frames=52, fps=FPS_JAHR, path=path)


---
## 9. GIF D — Cross-Border Import/Export<a id='gif_d_K_01c'></a>

[↑ TOC](#toc_K_01c)

Moving Dots zwischen CH-Zentrum und Grenzpunkten DE/AT/IT/FR. Richtung = Export (raus) / Import (rein).

In [ ]:
# ── GIF D: Cross-Border Tagesverlauf ─────────────────────────────────────────
CH_CENTER = (8.15, 46.90)   # Geographisches CH-Zentrum für Cross-Border-Flows

# Vereinfachtes Cross-Border-Modell:
# CH netto = Summe aller Zonen-Imbalance → verteilt auf Nachbarn nach Anteilsschlüssel
NEIGHBOR_WEIGHT = {'DE':0.35,'AT':0.15,'IT':0.30,'FR':0.20}  # ⚙ Anteil an Netto-Austausch

def get_border_flows(get_imb_fn):
    '''Berechne Grenzflüsse aus Gesamt-Imbalance.'''
    total_imb = sum(get_imb_fn(z) for z in ZONE_COLORS_B)
    return {nb: total_imb * w for nb, w in NEIGHBOR_WEIGHT.items()}

def update_gif_d(fig, ax, frame, n_frames, get_imb_fn, label):
    draw_base_map(ax, zone_alpha=0.06)
    border_flows = get_border_flows(get_imb_fn)

    for nb, mw in border_flows.items():
        draw_border_dots(ax, nb, mw, frame, n_frames, ch_center=CH_CENTER)
        # Grenzpunkt-Label
        bp = BORDER_POINTS[nb]
        ax.scatter(*bp, s=80, c=BORDER_COLORS[nb], alpha=0.95, zorder=12, linewidths=0)
        sign = '+' if mw > 0 else ''
        ax.text(bp[0], bp[1]+0.12, f"{nb}\n{sign}{mw:.0f} MW", ha='center', va='bottom',
                color=BORDER_COLORS[nb], fontsize=6, fontweight='bold', zorder=15,
                bbox=dict(boxstyle='round,pad=0.15', fc='#090d14', alpha=0.8, lw=0))

    # CH-Gesamtsaldo
    total = sum(border_flows.values())
    col_total = '#66BB6A' if total > 0 else '#EF5350'
    ax.scatter(*CH_CENTER, s=200, c=col_total, alpha=0.7, zorder=13, linewidths=0)
    ax.text(CH_CENTER[0], CH_CENTER[1]+0.18, f"CH netto\n{total:+.0f} MW",
            ha='center', va='bottom', color=col_total, fontsize=6.5, fontweight='bold',
            zorder=16, bbox=dict(boxstyle='round,pad=0.2', fc='#090d14', alpha=0.85, lw=0))

    handles = [mpatches.Patch(color=BORDER_COLORS[nb], label=f'CH↔{nb}') for nb in BORDER_POINTS]
    ax.legend(handles=handles, loc='lower left', fontsize=6,
              framealpha=0.5, facecolor='#111', labelcolor='white')
    add_timestamp_bar(fig, "⚙ Modelliert — vereinfachtes Netto-Modell",
                      label, "⚠️ Synthetische Daten")

for saison in SAISONS:
    def make_d_fn(s):
        def fn(fig, ax, frame, n_frames):
            get_imb = lambda z: float(HOURLY[s][z]['imb'][frame])
            update_gif_d(fig, ax, frame, n_frames, get_imb,
                         f"Cross-Border CH — {frame:02d}:00 Uhr ({s})")
        return fn
    path = os.path.join(CHARTS_DIR, f"kuer_k01c_gif_d_tag_{saison.lower()}.gif")
    make_gif(make_d_fn(saison), n_frames=24, fps=FPS_TAG, path=path)


In [ ]:
# ── GIF D: Cross-Border Jahresverlauf ────────────────────────────────────────
def update_gif_d_jahr(fig, ax, frame, n_frames):
    w = frame
    monat = MONAT_KURZ[int(w/52*12)]
    get_imb = lambda z: float(WEEKLY[z]['imb'][w])
    update_gif_d(fig, ax, frame, n_frames, get_imb,
                 f"Cross-Border CH — Woche {w+1:02d}/52 ({monat})")

path = os.path.join(CHARTS_DIR, 'kuer_k01c_gif_d_jahr.gif')
make_gif(update_gif_d_jahr, n_frames=52, fps=FPS_JAHR, path=path)


---
## 10. GIF E — Kombiniert (Zonenfluss + Cross-Border)<a id='gif_e_K_01c'></a>

[↑ TOC](#toc_K_01c)

GIF C + GIF D in einer einzigen Animation. Interne Flüsse + Grenzflüsse simultan.

In [ ]:
# ── GIF E: Kombiniert Tagesverlauf ───────────────────────────────────────────
def update_gif_e(fig, ax, frame, n_frames, get_imb_fn, label):
    '''GIF C + GIF D kombiniert.'''
    draw_base_map(ax, zone_alpha=0.09)

    # Zonenflüsse (GIF C)
    for zone_from, zone_to, _ in ZONE_PAIRS:
        imb_from = get_imb_fn(zone_from)
        imb_to   = get_imb_fn(zone_to)
        if imb_from > 0 and imb_to < 0:
            mw_flow = min(abs(imb_from), abs(imb_to))
            draw_flow_dots(ax, zone_from, zone_to, mw_flow, frame, n_frames)
        elif imb_to > 0 and imb_from < 0:
            mw_flow = min(abs(imb_to), abs(imb_from))
            draw_flow_dots(ax, zone_to, zone_from, mw_flow, frame, n_frames)
        else:
            net = imb_from - imb_to
            if abs(net) > 100:
                draw_flow_dots(ax, zone_from, zone_to, net, frame, n_frames)

    # Cross-Border-Flüsse (GIF D)
    border_flows = get_border_flows(get_imb_fn)
    for nb, mw in border_flows.items():
        draw_border_dots(ax, nb, mw, frame, n_frames, ch_center=CH_CENTER)
        bp = BORDER_POINTS[nb]
        ax.scatter(*bp, s=70, c=BORDER_COLORS[nb], alpha=0.95, zorder=12, linewidths=0)
        ax.text(bp[0], bp[1]+0.10, f"{nb}\n{mw:+.0f} MW", ha='center', va='bottom',
                color=BORDER_COLORS[nb], fontsize=5.5, fontweight='bold', zorder=15,
                bbox=dict(boxstyle='round,pad=0.12', fc='#090d14', alpha=0.75, lw=0))

    # Zone-Imbalance-Dots (klein, im Hintergrund)
    for zone, (cx, cy) in ZONE_CENTERS.items():
        imb = get_imb_fn(zone)
        col = '#66BB6A' if imb > 0 else '#EF5350'
        ax.scatter(cx, cy, s=max(40, min(250, abs(imb)/5)), c=col, alpha=0.50,
                   zorder=7, linewidths=0)

    add_zone_labels(ax, fontsize=6)
    # Kombinierte Legende
    handles = (
        [mpatches.Patch(color=ZONE_COLORS_B[z], label=z.replace('Mitte-','M-'))
         for z in list(ZONE_COLORS_B)[:3]] +
        [mpatches.Patch(color=BORDER_COLORS[nb], label=f'↔{nb}') for nb in BORDER_POINTS]
    )
    ax.legend(handles=handles, loc='lower left', fontsize=5.5, ncol=2,
              framealpha=0.5, facecolor='#111', labelcolor='white')
    add_timestamp_bar(fig, "⚙ Modelliert — Zonenfluss + Cross-Border",
                      label, "⚠️ Synthetische Daten")

for saison in SAISONS:
    def make_e_fn(s):
        def fn(fig, ax, frame, n_frames):
            get_imb = lambda z: float(HOURLY[s][z]['imb'][frame])
            update_gif_e(fig, ax, frame, n_frames, get_imb,
                         f"Energiefluss CH — {frame:02d}:00 Uhr ({s})")
        return fn
    path = os.path.join(CHARTS_DIR, f"kuer_k01c_gif_e_tag_{saison.lower()}.gif")
    make_gif(make_e_fn(saison), n_frames=24, fps=FPS_TAG, path=path)


In [ ]:
# ── GIF E: Kombiniert Jahresverlauf ──────────────────────────────────────────
def update_gif_e_jahr(fig, ax, frame, n_frames):
    w = frame
    monat = MONAT_KURZ[int(w/52*12)]
    get_imb = lambda z: float(WEEKLY[z]['imb'][w])
    update_gif_e(fig, ax, frame, n_frames, get_imb,
                 f"Energiefluss CH — Woche {w+1:02d}/52 ({monat})")

path = os.path.join(CHARTS_DIR, 'kuer_k01c_gif_e_jahr.gif')
make_gif(update_gif_e_jahr, n_frames=52, fps=FPS_JAHR, path=path)


---
## 11. Abschluss<a id='abschluss_K_01c'></a>

[↑ TOC](#toc_K_01c)

In [ ]:
# ── Abschlusskontrolle K_01c ─────────────────────────────────────────────────
print('K_01c – Abschlusskontrolle')
print('=' * 65)

expected = []
for saison in ['winter','frühling','sommer','herbst']:
    for gif in ['a','b','c','d','e']:
        expected.append(f'kuer_k01c_gif_{gif}_tag_{saison}.gif')
for gif in ['a','b','c','d','e']:
    expected.append(f'kuer_k01c_gif_{gif}_jahr.gif')

total_size = 0
ok_count = 0
for fname in expected:
    p = os.path.join(CHARTS_DIR, fname)
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p)//1024 if exists else 0
    icon = '✅' if exists else '❌'
    print(f"  {icon} {fname:<48} {size_kb:>5} KB")
    if exists:
        ok_count += 1
        total_size += size_kb

print()
print(f"  Total: {ok_count}/{len(expected)} GIFs | {total_size//1024:.1f} MB")
print()
print('⚙ Parameter (in diesem NB):')
print(f'  DPI_GIF={DPI_GIF} | FPS_TAG={FPS_TAG} | FPS_JAHR={FPS_JAHR}')
print()
print('Hinweis: Alle Flussdaten sind synthetisch modelliert.')
print('Für operative Analysen: Swissgrid SCADA oder ENTSO-E Transparency API.')


| [← K_01b Zonenmodell](K_01b_Zonenmodell_Erweitert.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|